In [2]:
!pip install -Uqq fastai icrawler


In [3]:
from fastai.vision.all import *
from icrawler.builtin import BingImageCrawler
import os, shutil, time

def download_to_folder(keyword, dest_folder, max_images=100):
    BingImageCrawler(storage={'root_dir': str(dest_folder)}).crawl(keyword=keyword, max_num=max_images)

path = Path('aiorreal')
if path.exists(): shutil.rmtree(path)

for name, query in [('ai', 'AI generated image artwork'), ('real', 'real photograph nature people')]:
    dest = path/name
    dest.mkdir(parents=True)
    download_to_folder(query, dest, max_images=80)
    time.sleep(5)
    resize_images(dest, max_size=400, dest=dest)

print(len(os.listdir(path/'ai')), "ai")
print(len(os.listdir(path/'real')), "real")

ERROR:downloader:Response status code 403, file https://static.vecteezy.com/system/resources/thumbnails/009/167/287/original/circuit-data-neural-network-ai-technology-cloud-computing-bits-internet-5g-blue-background-information-ai-talking-circuit-women-free-video.jpg


14 ai
2 real


In [4]:
failed = verify_images(get_image_files(path))
failed.map(Path.unlink)

[]

In [5]:
dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=[Resize(192, method='squish')]
).dataloaders(path, bs=8)

In [6]:
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(3)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 167MB/s]


epoch,train_loss,valid_loss,error_rate,time
0,1.478925,3.337117,0.333333,00:01


epoch,train_loss,valid_loss,error_rate,time
0,1.153843,2.661117,0.333333,00:03
1,1.307902,2.048237,0.333333,00:02
2,1.208031,1.517374,0.333333,00:02


In [7]:
learn.export('final_model.pkl')

In [8]:
import torch
torch.save(learn.model.state_dict(), 'model_weights.pth')
print(dls.vocab)

['ai', 'real']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')